In [0]:
import pandas as pd
from glob import glob
import os
from tqdm import tqdm
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

In [0]:
# ── Unity Catalog location where the resulting table will be saved ─────────
# See the README's "Key concepts" section for what catalog/schema mean.
CATALOG = "use1_prod_artemis_catalog_3718194974443840" # change
SCHEMA = "tier1_raw" # change
 
# Full name of the table that will be created/overwritten with the flight data.
TABLE_NAME = f"{CATALOG}.{SCHEMA}.drone_mission_table" # change
 
# ── Where the raw flight data lives ─────────────────────────────────────────
# root_path is the base folder that contains all the flights (organized in
# nested subfolders by site/trial/season/field/etc.). Update this to point
# to your own storage volume.
root_path = "/Volumes/use1_prod_artemis_catalog_3718194974443840/production/data/pheno_google/" # change
 
# This glob pattern walks 8 folder levels deep under root_path to find every
# "flight_details.json" file — one per flight. If your folder structure has
# a different number of levels, this pattern needs to be adjusted too.
flight_glob_path = f"{root_path}/*/*/*/*/*/*/*/*/flight_details.json" # change

In [0]:
# Find every flight_details.json file that matches the pattern above.
files = glob(flight_glob_path)
print(len(files))

6


In [0]:
row_list = []
 
# ── Build one row of metadata per flight ────────────────────────────────────
for fd in tqdm(files):
    # The flight's folder is just the parent folder of its flight_details.json.
    flight_path = os.path.dirname(fd)
 
    # Look for camera/sensor subfolders inside this flight's raw_data folder
    # (just used here to report how many camera folders were found).
    camera_path = f'{flight_path}/raw_data/*'
    camera_files = glob(camera_path)
    print(f"cameras: {len(camera_files)}")
 
    #for camera_file in camera_files:
 
    # Split the full file path into its individual folder names, so we can
    # pull out specific metadata fields by their position in the path.
    # This depends entirely on the folder structure matching the glob
    # pattern above — if your folder hierarchy differs, these index numbers
    # (7, 8, 9, etc.) will need to be updated to point at the correct folders.
    split_text = fd.split('/')
 
    #raw_path = f'{flight_path}/raw_data/{camera}/*'
    #raw_files = glob(raw_path)
 
    # Map each relevant path segment to a named field for this flight.
    row_dict = {'site': split_text[7],
                'trial': split_text[8],
                'season': split_text[9],
                'flight_date': split_text[14],
                'field': split_text[10],
                'location': split_text[11],
                'mission': split_text[13],
                'flight_metadata_path': fd}
 
    row_list.append(row_dict)

100%|██████████| 6/6 [00:00<00:00, 19.59it/s]

cameras: 0
cameras: 0
cameras: 0
cameras: 0
cameras: 0
cameras: 0


In [0]:
# ── Assemble and save the final table ───────────────────────────────────────
# Collect all the flight rows into a single pandas DataFrame first, for a
# quick local sanity check before writing it out as a Spark table.
flight_data_df = pd.DataFrame(row_list)
flight_data_df

,site,trial,season,flight_date,field,location,mission,flight_metadata_path
0,CIAT_CALI,Aduthurai,Paddy,2024_11_24_00_00,unknown,unknown,unknown,/Volumes/use1_prod_artemis_catalog_37181949744...
1,CIAT_CALI,Aduthurai,Paddy,2024_12_09_00_00,unknown,unknown,unknown,/Volumes/use1_prod_artemis_catalog_37181949744...
2,CIAT_CALI,Bhavanisagar,Tomato,2025_01_02_00_00,unknown,unknown,unknown,/Volumes/use1_prod_artemis_catalog_37181949744...
3,CIAT_CALI,Bhavanisagar,Tomato,2025_01_03_00_00,unknown,unknown,unknown,/Volumes/use1_prod_artemis_catalog_37181949744...
4,CIAT_CALI,Kovilpatti,Sorghum,2024_11_05_00_00,unknown,unknown,unknown,/Volumes/use1_prod_artemis_catalog_37181949744...
5,CIAT_CALI,Kovilpatti,Sorghum,2024_12_02_00_00,unknown,unknown,unknown,/Volumes/use1_prod_artemis_catalog_37181949744...


In [0]:
len(flight_data_df)

6

In [0]:
# Convert to a Spark DataFrame so it can be saved as a table in the catalog.
spark_df = spark.createDataFrame(flight_data_df)

spark_df.printSchema()

root
 |-- site: string (nullable = true)
 |-- trial: string (nullable = true)
 |-- season: string (nullable = true)
 |-- flight_date: string (nullable = true)
 |-- field: string (nullable = true)
 |-- location: string (nullable = true)
 |-- mission: string (nullable = true)
 |-- flight_metadata_path: string (nullable = true)



In [0]:
# Write (overwrite) the table in Unity Catalog. "mergeSchema" allows the
# table's schema to evolve if new columns are added in future runs.
spark_df.write.option("mergeSchema", "true").saveAsTable(TABLE_NAME, mode="overwrite")